In [16]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv.ipython import load_dotenv
import os

In [17]:
load_dotenv(override=True)

True

In [21]:
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
    
)

In [40]:
print(type(ChatGroq))

<class 'pydantic._internal._model_construction.ModelMetaclass'>


In [22]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helful assistant"

)

In [27]:
resp=agent.invoke(input={"messages": [

    {"role": "user", "content": "my name ise Rahma"}

]})


In [33]:
print(resp["messages"][-1].content)

Nice to meet you, Rahma! How can I assist you today?


In [45]:
resp1=agent.invoke(input={"messages": [
    {"role": "user", "content": "c'est quoi mon Nom"}
]})

In [46]:
print(resp1["messages"][-1].content)

Je ne connais pas votre nom, car je n’ai pas accès à vos informations personnelles.  
Si vous souhaitez que je m’adresse à vous d’une façon particulière, n’hésitez pas à me le dire !


In [78]:
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, AIMessage, SystemMessage

In [59]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest, handler)-> ModelResponse:
    env=request.runtime.context.get("env", "test")
    if env=="test":
        model =llm
    else:
        model = llm
    return handler(request.override(model=model))
    

In [76]:


agent2 = create_agent(
    model=llm,
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
   
)

In [77]:
resp2=agent2.invoke(
    input={"messages": [ {"role": "user", "content": "my name ise Rahma"}]},

    context = {"env": "test"}

)

[values] {'messages': [HumanMessage(content='my name ise Rahma', additional_kwargs={}, response_metadata={}, id='b48dd9c1-2a93-4e4c-8baf-e55914574ae0')]}
[updates] {'model': {'messages': [AIMessage(content='Nice to meet you, Rahma! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "my name ise Rahma". Probably they meant "my name is Rahma". The user is just stating their name. The assistant should respond politely, maybe ask how can help. No disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 76, 'total_tokens': 150, 'completion_time': 0.157196824, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.003732248, 'prompt_tokens_details': None, 'queue_time': 0.151782603, 'total_time': 0.160929072}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6b677c2caf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'

In [72]:
print(resp2["messages"][-1].content)

Nice to meet you, Rahma! How can I help you today?


In [73]:
resp2=agent2.invoke(
    input={"messages": [{"role": "user", "content": "c'est quoi mon nom"}]},
    context= {"env": "test"}
)

In [74]:
print(resp2["messages"][-1].content)

Je ne connais pas votre nom. Si vous souhaitez que je m’adresse à vous d’une façon particulière, n’hésitez pas à me le dire !


In [80]:
memory= InMemorySaver()
agent = create_agent(
    model=llm,
    system_prompt="You are a helful assistant",
    checkpointer=memory
)


In [84]:
config={"configurable":{"thread_id":1}}
resp3=agent.invoke(
    input={"messages": [HumanMessage("Je me nomme Rahma")]},
    config=config
)

In [85]:
print(resp3["messages"][-1].content)

Enchanté, Rahma ! Si vous avez la moindre question ou besoin d’aide, n’hésitez pas. 😊


In [89]:
resp3=agent.invoke(
    input={"messages": [HumanMessage("C'est quoi mon nom")]},
    config=config
)

In [90]:
print(resp3["messages"][-1].content)

Votre nom est Rahma.
